## Experiment 08
**Aim:** Build a Tic-Tac-Toe game using reinforcement learning in Python by using following tasks: a. Setting up the environment b. Defining the Tic-Tac-Toe game c. Building the reinforcement learning model d. Training the model e. Testing the model

In [ ]:
import tkinter as tk
import random

Q = {}
alpha, gamma, epsilon = 0.3, 0.9, 0.1

def winner(b):
    lines = [(0,1,2), (3,4,5), (6,7,8), (0,3,6),
             (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
    for a, c, d in lines:
        if b[a] == b[c] == b[d] != "":
            return b[a]
    return None

def empty(b):
    return [i for i in range(9) if b[i] == ""]

def choose(b, explore):
    moves = empty(b)
    if explore and random.random() < epsilon:
        return random.choice(moves)
    best = max(Q.get((tuple(b), m), 0) for m in moves)
    return random.choice([m for m in moves if Q.get((tuple(b), m), 0) == best])

print("Training AI...")
for episode in range(30000):
    board = [""] * 9
    history = []

    while True:
        move = choose(board, True)
        history.append((tuple(board), move))
        board[move] = "X"

        if winner(board) == "X":
            reward = 1
            break
        if not empty(board):
            reward = 0
            break

        board[random.choice(empty(board))] = "O"

        if winner(board) == "O":
            reward = -1
            break
        if not empty(board):
            reward = 0
            break

    for state, move in reversed(history):
        old = Q.get((state, move), 0)
        Q[(state, move)] = old + alpha * (reward - old)
        reward = reward * gamma

print("Training completed!")

root = tk.Tk()
root.title("RL Tic-Tac-Toe")
board = [""] * 9
buttons = []

def game_result():
    w = winner(board)
    if w == "X":
        return "AI wins!"
    if w == "O":
        return "You win!"
    if not empty(board):
        return "It's a draw!"
    return ""

def show():
    for i in range(9):
        buttons[i].config(text=board[i])
    label.config(text=game_result())

def ai_move():
    if game_result() != "":
        return
    board[choose(board, False)] = "X"
    show()

def click(i):
    if board[i] != "" or game_result() != "":
        return
    board[i] = "O"
    show()
    if game_result() == "":
        root.after(400, ai_move)

def new_game():
    for i in range(9):
        board[i] = ""
    show()

for i in range(9):
    b = tk.Button(root, text="", font=("Arial", 30, "bold"),
                  width=4, height=2, command=lambda i=i: click(i))
    b.grid(row=i // 3, column=i % 3)
    buttons.append(b)

label = tk.Label(root, text="", font=("Arial", 14))
label.grid(row=3, column=0, columnspan=3)

tk.Button(root, text="New Game", command=new_game).grid(
    row=4, column=0, columnspan=3)

root.mainloop()